# Duplicate search (Sentence-BERT + FAISS) and priority model

In [ ]:
!pip -q install sentence-transformers faiss-cpu
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

DRIVE = "/content/drive/MyDrive/issue-triage"
ART = f"{DRIVE}/artifacts"
os.makedirs(ART, exist_ok=True)
sys.path.insert(0, DRIVE)                     

EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ = 256
SEED = 42

## 1. Build the search corpus
All real issues in the dataset, plus the original issues that were downloaded because duplicates point to them.

In [ ]:
from preprocess import clean_issue

df = pd.read_parquet(f"{DRIVE}/issues_clean.parquet")
extra = pd.read_json(f"{DRIVE}/extra_corpus.jsonl", lines=True)

feats = pd.DataFrame([clean_issue(t, b) for t, b in zip(extra["title"], extra["body"])], index=extra.index)
extra = pd.concat([extra, feats], axis=1)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
extra["created_at"] = pd.to_datetime(extra["created_at"], utc=True)

cols = ["repo", "number", "created_at", "title", "text_clean"]
corpus = pd.concat([df[cols].assign(source="dataset"), extra[cols].assign(source="extra")], ignore_index=True)
corpus = corpus.drop_duplicates(["repo", "number"]).reset_index(drop=True)
print(len(df), "dataset issues +", len(extra), "extra originals ->", len(corpus), "in the corpus")
print(corpus.groupby("repo").size().to_dict())

## 2. Embed every issue with Sentence-BERT

In [ ]:
emb_path = f"{ART}/corpus_emb.npy"
if os.path.exists(emb_path) and len(np.load(emb_path, mmap_mode="r")) == len(corpus):
    emb = np.load(emb_path)
    print("loaded saved embeddings", emb.shape)
else:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(EMB_MODEL, device="cuda")
    st.max_seq_length = MAX_SEQ
    emb = st.encode(corpus["text_clean"].tolist(), batch_size=128, normalize_embeddings=True,
                    show_progress_bar=True).astype("float32")
    np.save(emb_path, emb)
    corpus.to_parquet(f"{ART}/corpus_meta.parquet")
    print("saved", emb.shape)

## 3. Evaluate duplicate retrieval
For each issue that was closed as a duplicate, search for its original **among earlier issues of the same repository** (a real bot only sees the past). We report recall@k (is the original in the top k?) and MRR, for Sentence-BERT and for a TF-IDF baseline.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

pairs = pd.read_csv(f"{DRIVE}/duplicate_pairs.csv")
key = {(r, n): i for i, (r, n) in enumerate(zip(corpus["repo"], corpus["number"]))}
pairs["q"] = [key.get((r, n)) for r, n in zip(pairs["repo"], pairs["number"])]
pairs["t"] = [key.get((r, n)) for r, n in zip(pairs["repo"], pairs["target"])]
pairs = pairs.dropna(subset=["q", "t"]).astype({"q": int, "t": int})

ts = corpus["created_at"].values
pairs["reachable"] = ts[pairs["t"].values] < ts[pairs["q"].values]       # original must be older than the duplicate
pairs["target_source"] = corpus["source"].values[pairs["t"].values]
print(len(pairs), "pairs in the corpus;", int((~pairs["reachable"]).sum()), "excluded because the original is newer than the duplicate")
P = pairs[pairs["reachable"]].reset_index(drop=True)

repo_rows = {r: np.where(corpus["repo"].values == r)[0] for r in corpus["repo"].unique()}

def ranks_for(score_fn):
    ranks = []
    for q, t, repo in zip(P["q"], P["t"], P["repo"]):
        cand = repo_rows[repo]
        cand = cand[(ts[cand] < ts[q]) & (cand != q)]
        s = score_fn(q, cand)
        ranks.append(1 + int((s > s[np.where(cand == t)[0][0]]).sum()))
    return np.array(ranks)

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=300_000)
X = tfidf.fit_transform(corpus["text_clean"])

ranks = {
    "TF-IDF cosine": ranks_for(lambda q, c: (X[c] @ X[q].T).toarray().ravel()),
    "Sentence-BERT": ranks_for(lambda q, c: emb[c] @ emb[q]),
}

subsets = {"all reachable pairs": np.ones(len(P), bool),
           "original already in dataset": (P["target_source"] == "dataset").values,
           "VS Code only": (P["repo"] == "microsoft/vscode").values,
           "pandas only": (P["repo"] == "pandas-dev/pandas").values}
rows = []
for method, r in ranks.items():
    for name, mask in subsets.items():
        x = r[mask]
        rows.append({"method": method, "subset": name, "pairs": int(mask.sum()),
                     "recall@1": (x <= 1).mean(), "recall@5": (x <= 5).mean(),
                     "recall@10": (x <= 10).mean(), "MRR": (1 / x).mean()})
results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))
results.to_csv(f"{ART}/retrieval_results.csv", index=False)

cs = [len(repo_rows[r]) for r in P["repo"]]
print("\nmedian candidate pool per query (same repo, earlier issues):", int(np.median(cs)))

## 4. Build the FAISS indexes used by the app (one per repository)

In [ ]:
import faiss
for repo, rows_idx in repo_rows.items():
    name = repo.replace("/", "__")
    index = faiss.IndexFlatIP(emb.shape[1])          # inner product on normalized vectors = cosine similarity
    index.add(emb[rows_idx])
    faiss.write_index(index, f"{ART}/faiss_{name}.index")
    np.save(f"{ART}/faiss_{name}_rows.npy", rows_idx)
    top = index.search(emb[rows_idx[:1]], 1)[1][0][0]
    print(repo, index.ntotal, "vectors | self-match check:", top == 0)
corpus.to_parquet(f"{ART}/corpus_meta.parquet")

## 5. Priority model (XGBoost)
Input: the Sentence-BERT embedding plus features known **when the issue is created** (code block, traceback, links, lengths, author association, repo). Comments and reactions are never used as inputs, only to build the label.

In [ ]:
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

LEVELS = ["low", "medium", "high"]
D = df[df["priority"].notna()].copy().reset_index(drop=True)
rows_in_corpus = np.array([key[(r, n)] for r, n in zip(D["repo"], D["number"])])
y = D["priority"].map({l: i for i, l in enumerate(LEVELS)}).values
split = D["split"].values
tr, va, te = split == "train", split == "val", split == "test"

assoc_cats = sorted(D["author_association"].fillna("NONE").unique())
repo_cats = sorted(D["repo"].unique())
meta_cols = ["has_code_block", "has_traceback", "n_urls", "n_error_names", "title_len", "body_len"]

def meta_features(frame):
    m = frame[meta_cols].astype(float).values
    a = pd.Categorical(frame["author_association"].fillna("NONE"), categories=assoc_cats).codes[:, None]
    r = pd.Categorical(frame["repo"], categories=repo_cats).codes[:, None]
    return np.hstack([m, a, r]).astype(float)

M = meta_features(D)
feature_sets = {"metadata only": M, "metadata + Sentence-BERT": np.hstack([M, emb[rows_in_corpus]])}

def fit_eval(X, name):
    model = xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.8,
                              colsample_bytree=0.5, tree_method="hist", objective="multi:softprob",
                              eval_metric="mlogloss", early_stopping_rounds=30, random_state=SEED, n_jobs=-1)
    model.fit(X[tr], y[tr], sample_weight=compute_sample_weight("balanced", y[tr]),
              eval_set=[(X[va], y[va])], verbose=False)
    out = {"model": name}
    for s, m_ in [("val", va), ("test", te)]:
        p = model.predict(X[m_])
        out[f"{s}_macro_f1"] = f1_score(y[m_], p, average="macro")
        out[f"{s}_accuracy"] = accuracy_score(y[m_], p)
        out[f"{s}_avg_level_error"] = float(np.abs(p - y[m_]).mean())
    return model, out

fitted, table = {}, []
for name, X_ in feature_sets.items():
    fitted[name], row = fit_eval(X_, name)
    table.append(row)
base = {"model": "always predict 'low'"}
for s, m_ in [("val", va), ("test", te)]:
    p = np.zeros(m_.sum(), int)
    base[f"{s}_macro_f1"] = f1_score(y[m_], p, average="macro", zero_division=0)
    base[f"{s}_accuracy"] = accuracy_score(y[m_], p)
    base[f"{s}_avg_level_error"] = float(np.abs(p - y[m_]).mean())
table.append(base)
print(pd.DataFrame(table).round(3).to_string(index=False))

best_name = max(table[:2], key=lambda r: r["val_macro_f1"])["model"]      # chosen on validation only
fitted[best_name].save_model(f"{ART}/priority_xgb.json")
json.dump({"model": best_name, "uses_embeddings": "Sentence" in best_name, "levels": LEVELS,
           "meta_cols": meta_cols, "assoc_cats": assoc_cats, "repo_cats": repo_cats},
          open(f"{ART}/priority_config.json", "w"), indent=1)
print("\nsaved priority model:", best_name)

In [ ]:
print("files in", ART)
for f in sorted(os.listdir(ART)):
    print(f"  {f}  ({os.path.getsize(os.path.join(ART, f)) / 1e6:.1f} MB)")